In [25]:
from pyspark.sql.functions import col,when,udf,round,expr,to_date
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, FloatType, DateType, LongType
from bs4 import BeautifulSoup
from pyspark.sql import SparkSession
import os
import json

In [26]:
spark = SparkSession.builder \
    .appName("SilverStep") \
    .config("spark.jars.packages", "org.apache.spark:spark-avro_2.12:3.5.0") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [27]:
current_path = os.getcwd()

dir_path = os.path.dirname(os.path.dirname(current_path))

with open(dir_path + '/steam/config.json', 'r') as arquivo:
  config = json.load(arquivo)

In [28]:
# Verificar o ambiente
if "dev" in dir_path:
    env = "dev"
elif "prod" in current_path:
    env = "prod"
else:
    env = "unknown"

print(f"A variável de ambiente é: {env}")

A variável de ambiente é: dev


In [29]:
schema = StructType([
    StructField("id", LongType(), True),
    StructField("file_date", DateType(), True),
    StructField("img", StringType(), True),
    StructField("name", StringType(), True),
    StructField("playtime", FloatType(), True),
    StructField("short_description", StringType(), True),
    StructField("website", StringType(), True),
    StructField("recommended", StringType(), True),
    StructField("minimum", StringType(), True),
    StructField("link", StringType(), True)
])

In [30]:
path = dir_path + "/datalake/steam/user/bronze/"
df = spark.read.format("avro").load(path)

In [31]:
df = df.select("appid", "file_date", "header_image", "name", "playtime_forever",
                "short_description","website","pc_requirements_recommended",
                "pc_requirements_minimum")

In [32]:
df = df.withColumnRenamed("playtime_forever", "playtime")\
    .withColumnRenamed("header_image", "img")

In [33]:
df = df.withColumn("playtime", round((col("playtime").cast("float") / 60), 2))

In [34]:
def extrair_texto_html(html):
    if html is None:
        return ""
    soup = BeautifulSoup(html, 'html.parser')
    texto = soup.get_text()
    return texto

extrair_texto_html_udf = udf(extrair_texto_html, StringType())

In [35]:
df = df.withColumn('pc_requirements_minimum', when((df['pc_requirements_minimum'].isNull()) | (df['pc_requirements_minimum'] == ""), "-").otherwise(extrair_texto_html_udf(df['pc_requirements_minimum'])))
df = df.withColumn('pc_requirements_recommended', when((df['pc_requirements_recommended'].isNull()) | (df['pc_requirements_recommended'] == ""), "-").otherwise(extrair_texto_html_udf(df['pc_requirements_recommended'])))

In [36]:
df = df.withColumn('pc_requirements_minimum', expr("regexp_replace(pc_requirements_minimum, 'pc_requirements_minimum:', '')"))
df = df.withColumn('pc_requirements_minimum', expr("regexp_replace(pc_requirements_minimum, 'pc_requirements_minimum', '')"))
df = df.withColumn('pc_requirements_recommended', expr("regexp_replace(pc_requirements_recommended, 'pc_requirements_recommended:', '')"))
df = df.withColumn('pc_requirements_recommended', expr("regexp_replace(pc_requirements_recommended, 'pc_requirements_recommended', '')"))

In [37]:
df = df.withColumn("link", F.concat(F.lit("https://store.steampowered.com/app/"), F.col("appid")))

In [38]:
df = df.withColumn("file_date", to_date(df["file_date"], "yyyy-MM-dd"))

In [39]:
df = spark.createDataFrame(df.rdd, schema)

In [40]:
string_cols = [f.name for f in df.schema.fields if f.dataType == StringType()]
for col_name in string_cols:
    df = df.withColumn(col_name, when(col(col_name).isNull(), '-').otherwise(col(col_name)))
    df = df.withColumn(col_name, when(col(col_name) == 'nan', '-').otherwise(col(col_name)))

In [41]:
df.show()

+-------+----------+--------------------+--------------------+--------+--------------------+--------------------+--------------------+--------------------+--------------------+
|     id| file_date|                 img|                name|playtime|   short_description|             website|         recommended|             minimum|                link|
+-------+----------+--------------------+--------------------+--------+--------------------+--------------------+--------------------+--------------------+--------------------+
|    730|2024-11-09|https://shared.ak...|    Counter-Strike 2|    1.03|For over two deca...|http://counter-st...|                   -|Minimum:OS: Windo...|https://store.ste...|
|   4700|2024-11-09|https://shared.ak...|Total War: MEDIEV...|   16.65|Spanning the most...|http://www.totalw...|                   -|Minimum System Re...|https://store.ste...|
|   4760|2024-11-09|https://shared.ak...|Rome: Total War™ ...|  766.78|Control and conqu...|http://www.totalw...|  

In [42]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- file_date: date (nullable = true)
 |-- img: string (nullable = true)
 |-- name: string (nullable = true)
 |-- playtime: float (nullable = true)
 |-- short_description: string (nullable = true)
 |-- website: string (nullable = true)
 |-- recommended: string (nullable = true)
 |-- minimum: string (nullable = true)
 |-- link: string (nullable = true)



In [43]:
output_path = dir_path + "/datalake/steam/user/silver/"
os.makedirs(output_path, exist_ok=True)

In [44]:
df.write.mode('overwrite').partitionBy('file_date').parquet(output_path)